<a href="https://colab.research.google.com/github/ansonkwokth/TableTennisPrediction/blob/dev/Siamese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/ansonkwokth/TableTennisPrediction.git
%cd TableTennisPrediction

fatal: destination path 'TableTennisPrediction' already exists and is not an empty directory.
/content/TableTennisPrediction


In [3]:

import pandas as pd
from utils import data_loader as dl

import numpy as np
from model.Elo import Elo
from model.ModifiedElo import ModifiedElo
from model.ensemble import BaggingRatingSystem

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

import copy
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# Data

In [44]:
GAME = 'TTStar'
# GAME = 'TTCup'
# GAME = 'SetkaCup'
# GAME = 'SetkaCupWomen'
# GAME = 'LigaPro'


In [45]:
match GAME:
    case 'TTStar':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'TTCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCupWomen':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'LigaPro':
        years = [2022, 2023, 2024]
    case _:
        raise ValueError("Invalid game selected.")


text_data_game = dl.load_game_data(GAME, years, '../')
text_data = {
    year: text_data_game[year] for year in years
}
df = dl.create_game_dfs(GAME, years, text_data)

Loading ..//TTStar2020.txt
Loading ..//TTStar2021.txt
Loading ..//TTStar2022.txt
Loading ..//TTStar2023.txt
Loading ..//TTStar2024.txt


In [46]:
# Generate ID indices for each pair of rows in the DataFrame
idx_lt = [i for i in range(len(df) // 2) for _ in range(2)]
df['ID'] = idx_lt  # Assign to the 'ID' column

# Reset the DataFrame index to ensure it's sequential
df.reset_index(drop=True, inplace=True)

# Get unique players and store them in player_lt
player_lt = df['Player'].unique()



In [47]:
year_val = years[-2]
year_test = years[-1]

df_train = df.loc[pd.DatetimeIndex(df['Date']).year < year_val]
df_val = df.loc[pd.DatetimeIndex(df['Date']).year == year_val]
df_test = df.loc[pd.DatetimeIndex(df['Date']).year == year_test]

In [48]:
def format_to_array(df: pd.DataFrame) -> np.ndarray:

    # info_col = ['ID', 'Round', 'Datetime', 'Game', 'Date', 'Time']
    info_col = ['Round', 'Datetime', 'Game', 'Date', 'Time']
    col = [item for item in df.columns if item not in info_col]

    df[[c for c in col if "Set" in c]] = df[[c for c in col if "Set" in c]].astype(float)
    X = df[col].values.reshape(-1, 2, len(col))
    return X

In [49]:
X_train = format_to_array(df_train)
X_val = format_to_array(df_val)
X_test = format_to_array(df_test)

In [50]:
X_all = format_to_array(df)

In [51]:
X_all

array([[[0, 'Reitspies D.', 11.0, ..., nan, nan, nan],
        [0, 'Gavlas A.', 9.0, ..., nan, nan, nan]],

       [[1, 'Kleprlik J.', 3.0, ..., nan, nan, nan],
        [1, 'Prokopcov D.', 11.0, ..., nan, nan, nan]],

       [[2, 'Horejsi M.', 11.0, ..., 7.0, 9.0, nan],
        [2, 'Tregler T.', 4.0, ..., 11.0, 11.0, nan]],

       ...,

       [[17254, 'Reitspies D.', 5.0, ..., 11.0, 9.0, nan],
        [17254, 'Shetty S.', 11.0, ..., 9.0, 11.0, nan]],

       [[17255, 'Cappuccio M.', 5.0, ..., 3.0, nan, nan],
        [17255, 'Tymofieiev O.', 11.0, ..., 11.0, nan, nan]],

       [[17256, 'Lakatos T.', 11.0, ..., nan, nan, nan],
        [17256, 'Cappuccio C.', 9.0, ..., nan, nan, nan]]], dtype=object)

In [52]:
def get_data(X):
    data_all = []
    for game in X:
        game = game[:, 1:]
        player1, player2 = game[:, 0]
        scores = game[:, 1:]

        for si in scores.T:
            if (si[0] + si[1]) == 0: continue
            ti = si[0] / (si[0] + si[1])
            if not np.isnan(ti):
                data_all.append([player1, player2, ti])
    return data_all

def get_data_idx(data, player_to_idx):
    data_all_idx = []
    for game in data:
        data_all_idx.append((player_to_idx[game[0]], player_to_idx[game[1]], game[2]))
    return data_all_idx

data_all = get_data(X_all)
data_train = get_data(X_train)
data_test = get_data(X_test)
player_to_idx = {pn: i for i, pn in enumerate(np.unique(np.array(data_all)[:, :2]))}


data_all_idx = get_data_idx(data_all, player_to_idx)
data_train_idx = get_data_idx(data_train, player_to_idx)
data_test_idx = get_data_idx(data_test, player_to_idx)

In [5]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class TableTennisSetScoreDataset(Dataset):
    def __init__(self, data_array, player_to_idx, score_start=2):
        """
        data_array: NumPy array of shape (n_games, 2, n_features)
          where each row is [game_index, player_name, set_score1, set_score2, ...]
        player_to_idx: dict mapping player names to unique indices.
        score_start: the column index where set scores start.
        """
        self.data_array = data_array
        self.player_to_idx = player_to_idx
        self.score_start = score_start
        # Assume that all games have the same number of score columns.
        self.max_sets = data_array.shape[2] - score_start

    def __len__(self):
        return self.data_array.shape[0]

    def __getitem__(self, idx):
        # Get the game record (2 rows, one for each player)
        game = self.data_array[idx]

        # Extract rows for player1 and player2.
        # Each row: [game_index, player_name, set_score1, set_score2, ...]
        p1_row = game[0]
        p2_row = game[1]

        # Extract player names and convert to indices.
        p1_name = p1_row[1]
        p2_name = p2_row[1]
        p1_idx = self.player_to_idx[p1_name]
        p2_idx = self.player_to_idx[p2_name]

        # Extract set scores for each player.
        p1_scores = np.array(p1_row[self.score_start:], dtype=np.float32)
        p2_scores = np.array(p2_row[self.score_start:], dtype=np.float32)

        # Create a mask for valid sets (assume both players have valid scores for the same sets).
        # True where the score is not NaN.
        mask = ~np.isnan(p1_scores)

        # Replace NaN values with 0 for numerical stability.
        p1_scores = np.nan_to_num(p1_scores, nan=0.0)
        p2_scores = np.nan_to_num(p2_scores, nan=0.0)

        # Convert everything to torch tensors.
        p1_idx = torch.tensor(p1_idx, dtype=torch.long)
        p2_idx = torch.tensor(p2_idx, dtype=torch.long)

        p1_scores = torch.tensor(p1_scores, dtype=torch.float)
        p2_scores = torch.tensor(p2_scores, dtype=torch.float)
        mask = torch.tensor(mask.astype(np.float32), dtype=torch.float)

        return p1_idx, p2_idx, p1_scores, p2_scores, mask

def create_player_mapping(data_array):
    """
    Build a dictionary mapping player names to unique indices.
    """
    players = set()
    for game in data_array:
        players.add(game[0][1])
        players.add(game[1][1])

    player_to_idx = {player: idx for idx, player in enumerate(sorted(players))}
    return player_to_idx



# Create the player mapping.
player_to_idx = create_player_mapping(X_all)

# Create the dataset and DataLoader.
dataset = TableTennisSetScoreDataset(X_all, player_to_idx)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# # Example iteration over the DataLoader.
# for batch in dataloader:
#     p1_idx, p2_idx, p1_scores, p2_scores, mask = batch
#     print("Player1 indices:", p1_idx)
#     print("Player2 indices:", p2_idx)
#     print("Player1 set scores:\n", p1_scores)
#     print("Player2 set scores:\n", p2_scores)
#     print("Mask (valid sets):\n", mask)
#     sfd
#     print()

NameError: name 'X_all' is not defined

In [55]:
import torch
import torch.nn as nn

class Siamese(nn.Module):
    def __init__(self, num_players, embedding_dim):
        """
        Args:
            num_players (int): Number of unique players.
            embedding_dim (int): Dimension of the player embedding.
        """
        super(Siamese, self).__init__()
        # Embedding layer to convert player indices to dense vectors.
        self.embedding = nn.Embedding(num_players, embedding_dim)
        # Symmetric branch: processes the sum of embeddings.
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(embedding_dim, 1),
            # nn.ReLU(),
            # nn.Linear(8, 4),
            # nn.ReLU(),
            # nn.Linear(4, 1),
        )

    def forward(self, p):
        # Look up player embeddings.
        emb = self.embedding(p)
        logits = self.linear_relu_stack(emb)
        return logits

In [68]:
def my_loss(pred_prob, p1_scores, p2_scores, mask):
    """
    Computes the Mean Squared Error loss only over the valid set scores.

    Args:
        pred (Tensor): Predicted scores, shape [batch, max_sets].
        target (Tensor): Ground truth scores, shape [batch, max_sets].
        mask (Tensor): Mask with 1 for valid sets and 0 for missing sets, shape [batch, max_sets].
    """
    scores_sum = p1_scores + p2_scores
    t1 = p1_scores / scores_sum
    loss1 = t1 * torch.log(pred_prob)
    loss2 = (1 - t1) * torch.log(1 - pred_prob)

    loss = (loss1 + loss2) * mask
    loss = torch.nan_to_num(loss, nan=0.0)


    return - loss.sum()


In [62]:


# Forward pass.
# model('Slashova M.')
# model(torch.tensor(0))


In [70]:

num_players = len(player_to_idx)        # For example, 20 unique players.
embedding_dim = 16      # Embedding vector size.

# Initialize the model.

model = SiameseNetwork(num_players, embedding_dim=1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 1

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    ii = 0
    for batch in dataloader:
        # Unpack the batch.
        # p1_idx, p2_idx: player indices (shape: [batch])
        # p1_scores, p2_scores: set scores (shape: [batch, max_sets])
        # mask: binary mask for valid set scores (shape: [batch, max_sets])
        p1_idx, p2_idx, p1_scores, p2_scores, mask = batch

        optimizer.zero_grad()

        # Forward pass.
        pred_prob = model(p1_idx, p2_idx)

        print(p1_idx, p2_idx)
        print(p1_scores, p2_scores)
        print(pred_prob)

        # Compute losses for each player's predicted set scores.
        loss = my_loss(pred_prob, p1_scores, p2_scores, mask)
        print(loss)

        # Backward pass and optimization.
        loss.backward()
        optimizer.step()
        # loss = my_loss(pred_score1, pred_score2, p1_scores, p2_scores, mask)
        # print(loss)
        print()
        total_loss += loss.item()


        ii += 1
        if ii == 2: break


    print()
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

tensor([18]) tensor([28])
tensor([[ 5.,  9., 11., 11., 11.,  0.]]) tensor([[11., 11.,  4.,  5.,  3.,  0.]])
tensor([0.3550], grad_fn=<MulBackward0>)
tensor(3.9653, grad_fn=<NegBackward0>)

tensor([94]) tensor([205])
tensor([[4., 6., 9., 0., 0., 0.]]) tensor([[11., 11., 11.,  0.,  0.,  0.]])
tensor([nan], grad_fn=<MulBackward0>)
tensor(-0., grad_fn=<NegBackward0>)


Epoch 1/1, Loss: 0.0001


In [64]:
model(torch.tensor(0))

tensor([nan], grad_fn=<ViewBackward0>)

In [13]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Step 1: Simulate 10 players
# -----------------------------
num_players = 100
player_names = [f"player{i}" for i in range(num_players)]
players = {}
for i in range(num_players):
    # Each player gets a fixed Gaussian ability (mean and std)
    mean = np.random.uniform(50, 100)
    std = np.random.uniform(5, 15)
    players[i] = {"name": player_names[i], "mean": mean, "std": std}


In [34]:

# -----------------------------
# Step 2: Simulate games and sets
# -----------------------------
def simulate_set(player1_id, player2_id):
    """Simulate one set (first to 11 points) between two players."""
    p1 = players[player1_id]
    p2 = players[player2_id]
    score1, score2 = 0, 0
    while score1 < 11 and score2 < 11:
        sample1 = np.random.normal(p1["mean"], p1["std"])
        sample2 = np.random.normal(p2["mean"], p2["std"])
        if sample1 > sample2:
            score1 += 1
        else:
            score2 += 1
    return score1, score2

def simulate_game(player1_id, player2_id):
    """Simulate a game (best of 5 sets: first to win 3 sets) between two players.
       Returns a list of set results as tuples (score1, score2)."""
    sets = []
    wins1, wins2 = 0, 0
    while wins1 < 3 and wins2 < 3:
        s1, s2 = simulate_set(player1_id, player2_id)
        sets.append((s1, s2))
        if s1 > s2:
            wins1 += 1
        else:
            wins2 += 1
    return sets

# Generate 100 games. For every set, record (player1_id, player2_id, t)
# where t = (points of player1) / (total points in the set)
data = []
for _ in range(100):
    player1_id, player2_id = np.random.choice(num_players, size=2, replace=False)
    sets = simulate_game(player1_id, player2_id)
    for s1, s2 in sets:
        t = s1 / (s1 + s2)  # e.g., if the set ended 11:9 then t = 11/20 = 0.55
        data.append((player1_id, player2_id, t))


In [53]:

# -----------------------------
# Step 3: Prepare the Dataset and DataLoader
# -----------------------------
class TableTennisDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        p1, p2, t = self.data[idx]
        return int(p1), int(p2), t

# dataset = TableTennisDataset(data)
dataset = TableTennisDataset(data_all_idx)
dataset_train = TableTennisDataset(data_train_idx)
dataloader_train = DataLoader(dataset_train, batch_size=16, shuffle=True)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)


In [54]:

# -----------------------------
# Step 4: Build the Integrated Siamese Network
# -----------------------------
class SiameseNetwork(nn.Module):
    def __init__(self, num_players, embedding_dim=8):
        super(SiameseNetwork, self).__init__()
        # Shared embedding layer for players.
        self.embedding = nn.Embedding(num_players, embedding_dim)
        # A small MLP to convert the embedding into a scalar "score."
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 1)
        )

    def forward(self, player1_idx, player2_idx):
        # Process both players using the same embedding and MLP.
        embed1 = self.embedding(player1_idx)
        embed2 = self.embedding(player2_idx)
        score1 = self.fc(embed1).squeeze(-1)
        score2 = self.fc(embed2).squeeze(-1)
        # Directly return the win probability for player1.
        prob = 1.0 / (1.0 + torch.exp(score2 - score1))
        return prob


In [55]:
def loss_fn(p, t):
    epsilon = 1e-7
    p = torch.clamp(p, epsilon, 1 - epsilon)
    loss = - (t * torch.log(p) + (1 - t) * torch.log(1 - p))
    return loss.mean()


num_players = len(player_to_idx)
model = SiameseNetwork(num_players, embedding_dim=16)
optimizer = optim.Adam(model.parameters(), lr=0.001)


num_epochs = 10
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in dataloader_train:
        # Use zip(*) to properly unpack the batch into separate tuples
        player1_idx, player2_idx, t_val = batch
        player1_idx = player1_idx.long()
        player2_idx = player2_idx.long()
        t_val = t_val.float()

        p = model(player1_idx, player2_idx)
        loss = loss_fn(p, t_val)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}")


Epoch 1/10, Loss: 0.3898
Epoch 2/10, Loss: 0.3884
Epoch 3/10, Loss: 0.3882
Epoch 4/10, Loss: 0.3881
Epoch 5/10, Loss: 0.3881
Epoch 6/10, Loss: 0.3881
Epoch 7/10, Loss: 0.3880
Epoch 8/10, Loss: 0.3880
Epoch 9/10, Loss: 0.3880
Epoch 10/10, Loss: 0.3880


In [58]:

model.eval()  # set model to evaluation mode
correct = 0
total = 0

with torch.no_grad():
    for batch in X_train:
        player1, player2 = batch[:, 1]

        player1_idx, player2_idx = player_to_idx[player1], player_to_idx[player2]
        player1_idx = torch.tensor(player1_idx)
        player2_idx = torch.tensor(player2_idx)
        win1 = (sum(batch[0, 2:]>batch[1, 2:]))
        win2 = (sum(batch[0, 2:]<batch[1, 2:]))

        # Ground truth: 1 if player1 won the set, 0 otherwise.
        ground_truth = (win1 > win2)
        # Predicted probability from the model.
        p = model(player1_idx, player2_idx)
        prediction = (p > 0.5).float()  # threshold at 0.5
        # print(int(prediction == ground_truth), prediction, ground_truth)
        # sdf
        correct += int(prediction == ground_truth)
        total += 1

accuracy = correct / total
print("Test accuracy: {:.2f}%".format(accuracy * 100))

Test accuracy: 75.42%


In [43]:
# -----------------------------
# Step 7: Generate Testing Set
# -----------------------------
# We simulate additional games to create a testing set.
test_data = []
for _ in range(100):  # simulate 30 games
    player1_id, player2_id = np.random.choice(num_players, size=2, replace=False)
    sets = simulate_game(player1_id, player2_id)
    for s1, s2 in sets:
        t = s1 / (s1 + s2)
        test_data.append((player1_id, player2_id, t))

test_dataset = TableTennisDataset(test_data)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# -----------------------------
# Step 8: Evaluate Accuracy on the Test Set
# -----------------------------
# For evaluation, we consider player1 to have won the set if t > 0.5.
# Similarly, if the model's predicted probability p > 0.5, we predict a win for player1.
model.eval()  # set model to evaluation mode
correct = 0
total = 0

with torch.no_grad():
    for batch in test_dataloader:
        player1_idx, player2_idx, t_val = batch
        player1_idx = player1_idx.long()
        player2_idx = player2_idx.long()
        t_val = t_val.float()
        # Ground truth: 1 if player1 won the set, 0 otherwise.
        ground_truth = (t_val > 0.5).float()
        # Predicted probability from the model.
        p = model(player1_idx, player2_idx)
        prediction = (p > 0.5).float()  # threshold at 0.5
        correct += (prediction == ground_truth).sum().item()
        total += t_val.size(0)

accuracy = correct / total
print("Test accuracy: {:.2f}%".format(accuracy * 100))

Test accuracy: 81.19%
